In [ ]:
csv_file_path = './data/Hackathon: VizAvalanche/sample_5k.csv'

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# 1. Load data from the csv_file_path
df = pd.read_csv(csv_file_path)

family_col = 'ncbi_family'
genus_col = 'ncbi_genus'

# 3. Aggregate data for Treemap
# Level 1: Family totals
df_family = df.groupby(family_col).size().reset_index(name='count')

# Level 2: Genus nested in Family
df_genus = df.groupby([family_col, genus_col]).size().reset_index(name='count')

# 4. Construct the Hierarchy for Plotly
ids = []
labels = []
parents = []
values = []

# Add Families (Roots)
for _, row in df_family.iterrows():
    family_name = str(row[family_col])
    ids.append(family_name)
    labels.append(f"<b>{family_name}</b>")
    parents.append("")  # No parent for the root level
    values.append(row['count'])

# Add Genera  (Children)
for _, row in df_genus.iterrows():
    family_name = str(row[family_col])
    genus_name = str(row[genus_col])

    # We create a unique ID to prevent conflicts if two families share a genus name
    ids.append(f"{family_name}_{genus_name}")
    labels.append(genus_name)
    parents.append(family_name)
    values.append(row['count'])

# 5. Create figure
fig = go.Figure(go.Treemap(
    ids=ids,
    labels=labels,
    parents=parents,
    values=values,
    branchvalues="total",
    marker=dict(
        colorscale=[[0, "#58A4B0"], [1, "#E37B40"]], # VirJenDB Branding
    ),
    hovertemplate='<b>%{label}</b><br>Sequences: %{value}<extra></extra>'
))

fig.update_layout(
    title="<b>Taxonomic Diversity: Family > Genus Hierarchy</b>",
    template='plotly_white',
    margin=dict(t=50, l=10, r=10, b=10)
)

# 6. Plot
fig.show()